In [31]:
import sched_helpers
import pandas as pd 
import numpy as np
import pytz
import datetime

In [32]:
TZ = pytz.timezone('America/Detroit')
CALENDAR_URL = 'http://www.shiftadmin.com/schedule_ical_group.php?cd=UIwfTiYhARsmldQIKdk1addmZLRORGLhbHKREh1COb8%3D&gfs=g9,f1,f2,f3&local=0&vc=1'
SCHED_ICAL_START_DATE = TZ.localize(datetime.datetime(2021, 7, 1, 0, 0, 0)).astimezone(pytz.utc)
SCHED_ICAL_END_DATE = TZ.localize(datetime.datetime(2022, 6, 30, 23, 59, 59)).astimezone(pytz.utc)
RESIDENTS_CSV = 'residents.csv'
MASTER_BLOCK_SCHEDULE_CSV = 'master_block_schedule.csv'
OFF_SERVICE_HOURS_CSV = 'off_service_hours.csv'

In [80]:
osh = sched_helpers.off_service_hours_df('off_service_hours.csv')
s = sched_helpers.download_ical(CALENDAR_URL)
sched = sched_helpers.ical_to_df(s, start=SCHED_ICAL_START_DATE, end=SCHED_ICAL_END_DATE, tz=TZ)

CPU times: user 3.39 ms, sys: 604 µs, total: 4 ms
Wall time: 313 ms
CPU times: user 480 ms, sys: 0 ns, total: 480 ms
Wall time: 480 ms


In [69]:
sched.head()

,summary,resident,shift,start,end,type,facility
start,,,,,,,
2022-04-01 06:00:00-04:00,SJ SA S Jamali,S Jamali,SA,2022-04-01 10:00:00+00:00,2022-04-01 20:00:00+00:00,"EM 23 (SA, SB, SD)",St. Joseph Mercy Hospital
2022-04-01 06:00:00-04:00,SJ SH A Flessel,A Flessel,SH,2022-04-01 10:00:00+00:00,2022-04-01 20:00:00+00:00,"PGY 1 (UH, UI, SH, SI)",St. Joseph Mercy Hospital
2022-04-01 06:00:00-04:00,SJ SH C Millman,C Millman,SH,2022-04-01 10:00:00+00:00,2022-04-01 20:00:00+00:00,"PGY 1 (UH, UI, SH, SI)",St. Joseph Mercy Hospital
2022-04-01 06:00:00-04:00,SJ SH A Rimawi,A Rimawi,SH,2022-04-01 10:00:00+00:00,2022-04-01 20:00:00+00:00,"PGY 1 (UH, UI, SH, SI)",St. Joseph Mercy Hospital
2022-04-01 07:00:00-04:00,HMC HA M Hussain,M Hussain,HA,2022-04-01 11:00:00+00:00,2022-04-01 21:00:00+00:00,Adult,Hurley


In [35]:
sched.columns

Index(['summary', 'resident', 'shift', 'start', 'end', 'type', 'facility'], dtype='object')

In [77]:
%%time
mbs = pd.read_csv('master_block_schedule.csv', header=[0,1,2,3], index_col=0)
mbs = mbs.T.reset_index().drop(['block','week','week_end'], axis=1)
mbs.index = pd.period_range(mbs.loc[0, 'week_start'], freq='7D', periods=len(mbs), name='week')
mbs = mbs.drop('week_start', axis=1)
mbs = mbs.resample('D').ffill()
mbs.index = mbs.index.rename('day')
mbs = (mbs.reset_index()
          .melt(id_vars='day', var_name='resident', value_name='rotation')
          .set_index('day'))

# covert the mbs df in to a sched-like df
mbs['start'] = pd.to_timedelta(mbs['rotation'].replace(osh['start'].to_dict()))
mbs['end'] = pd.to_timedelta(mbs['rotation'].replace(osh['end'].to_dict()))
mbs['start'] = mbs.index.to_timestamp() + mbs['start']
mbs['end'] = mbs.index.to_timestamp() + mbs['end']
mbs = mbs[mbs['rotation'] != 'ED']
mbs['summary'] = 'OS ' + mbs['rotation'] + ' ' + mbs['resident']
mbs['shift'] = mbs['rotation']
mbs['type'] = 'Off Service'
mbs['facility'] = 'NA'
mbs = mbs[sched.columns]
mbs['start'] = mbs['start'].dt.tz_localize(TZ)
mbs['end'] = mbs['end'].dt.tz_localize(TZ)
mbs = mbs.reset_index(drop=True).set_index('start').sort_index()

CPU times: user 44 ms, sys: 19.7 ms, total: 63.7 ms
Wall time: 56.9 ms


In [67]:
mbs.set_index('day').index.to_timestamp() + mbs['start']

0      2022-04-07 06:00:00
1      2022-04-08 06:00:00
2      2022-04-09 06:00:00
3      2022-04-10 06:00:00
4      2022-04-11 06:00:00
               ...        
5371   2022-06-25 00:00:00
5372   2022-06-26 00:00:00
5373   2022-06-27 00:00:00
5374   2022-06-28 00:00:00
5375   2022-06-29 00:00:00
Length: 5376, dtype: datetime64[ns]

In [13]:
pd.period_range(mbs.loc[0,'week_start'], freq='7D', end='mbs.'=len(mbs))

13 12
